# Descarga De Datos

>ESTO SE PUEDE BORRAR

In [ ]:
from wget import download
import pandas as pd
from os import path

if not path.exists("results.zip"):
    download("https://media.githubusercontent.com/media/StackExchange/Survey/6ae2209cd35a55e92739ba82f7887fb86807a069/packages/archive/2025/results.csv")
else:
    print("Archivo Ya Descargado")


if not path.exists("schema.csv"):
    download("https://media.githubusercontent.com/media/StackExchange/Survey/refs/heads/main/packages/archive/2025/schema.csv")
else:
    print("Archivo Ya Descargado")

df_results = pd.read_csv("results.zip", low_memory=False)
df_schema = pd.read_csv("schema.csv", low_memory=False)

# schema.csv: 
**Es el diccionario de datos de la encuesta. results.csv tiene las respuestas de cada persona, pero con nombres de columna abreviados.El schema te dice qué pregunta exacta corresponde a cada una de esas columnas.**

# results.csv:
**Contiene las respuestas individuales de la "Stack Overflow Developer Survey 2025", una encuesta anual que Stack Overflow realiza a la comunidad de desarrolladores sobre su perfil, trabajo, tecnologías y uso de herramientas de IA.**

# Visión general

**Dimensiones**

In [8]:
filas, columnas = df_results.shape
print(f"El Dataset \"results\" tiene {filas:,} filas y {columnas} columnas")
print(f"Memoria: {df_results.memory_usage(deep=True).sum() / 1e6:.1f} MB")
#df_results.head()

El Dataset "results" tiene 49,191 filas y 172 columnas
Memoria: 375.2 MB


**Tipos De Datos**

In [9]:
df_results.info(verbose=True, show_counts=True)
print("\n")
tipos = (df_results.dtypes.astype(str).value_counts().rename_axis("Tipo de dato").reset_index(name="Cantidad de columnas"))
tipos


<class 'pandas.DataFrame'>
RangeIndex: 49191 entries, 0 to 49190
Data columns (total 172 columns):
 #    Column                                    Non-Null Count  Dtype  
---   ------                                    --------------  -----  
 0    ResponseId                                49191 non-null  int64  
 1    MainBranch                                49191 non-null  str    
 2    Age                                       49191 non-null  str    
 3    EdLevel                                   48149 non-null  str    
 4    Employment                                48339 non-null  str    
 5    EmploymentAddl                            44875 non-null  str    
 6    WorkExp                                   42893 non-null  float64
 7    LearnCodeChoose                           46858 non-null  str    
 8    LearnCode                                 33556 non-null  str    
 9    LearnCodeAI                               45201 non-null  str    
 10   AILearnHow                     

,Tipo de dato,Cantidad de columnas
0,str,119
1,float64,52
2,int64,1


**Identificador, Duplicados Y Columnas Sin Valor**

In [10]:
print("ResponseId es único?: ", df_results['ResponseId'].is_unique)
print("Filas duplicadas:", df_results.duplicated().sum())
print("Columnas con un solo valor distinto:",[c for c in df_results.columns if df_results[c].nunique(dropna=True) <= 1])

ResponseId es único?:  True
Filas duplicadas: 0
Columnas con un solo valor distinto: []


# Interpretación

**El dataset `results` contiene 49.191 respuestas (filas) y 172 variables (columnas), con un uso de memoria de aproximadamente 375 MB.**

- **Tipos de datos: 119 columnas son de texto (`str`), 52 son numéricas de punto flotante (`float64`) y 1 es entera (`int64`, el identificador `ResponseId`). Las numéricas son `float` y no `int` porque contienen valores nulos.**
- **Identificador: `ResponseId` es único, por lo que cada fila representa una respuesta distinta.**
- **Duplicados: no hay filas duplicadas.**
- **Columnas sin información: no hay columnas constantes ni vacías.**


## Previo A La Caracterización De Atributos...

**Falta de contexto en los nombres abreviados: La base `results.csv` utiliza códigos abreviados como nombres de columna (por ejemplo, `AIAcc`, `TechEndorse_1`, `JobSatPoints_3`). Sin el diccionario de datos, es imposible saber la naturaleza exacta de la pregunta o qué representa la respuesta.**

**Determinación precisa del tipo de variable: Conocer el enunciado exacto y la estructura original de la pregunta en `schema.csv` permite definir objetivamente si un atributo es cualitativo ordinal, nominal, o cuantitativo continuo, evitando clasificaciones incorrectas basadas únicamente en el tipo de dato de Pandas (float64 o object).**

**Insumo para las etapas de Limpieza e Hipótesis: Saber qué variables corresponden a preguntas obligatorias, opcionales o condicionales (matriztadas) justifica el porcentaje de valores nulos observado y guía las decisiones de imputación o filtrado para el planteo de hipótesis.**



# Vinculacion Entre `results` Y `schema`

**Para determinar qué atributo de `schema.csv` actúa como clave de diccionario para los nombres de las columnas de `results.csv`, se realizó una comprobación evaluando la intersección entre las 172 columnas de la base de datos de respuestas y los valores únicos de cada campo de la base de esquemas**

In [11]:

# Comprobación de coincidencia entre las columnas de "results" y las columnas del "schema"
cols_results = set(df_results.columns)

for col in df_schema.columns:
    valores_schema = set(df_schema[col].dropna().astype(str))
    coincidencias = cols_results.intersection(valores_schema)
    print(f"Coincidencias de las columnas de 'results' con df_schema['{col}']: {len(coincidencias)} de {len(cols_results)}")


# Identificación de columnas de results que no coinciden directamente con qname
cols_no_coincidentes = sorted(list(set(df_results.columns) - set(df_schema["qname"])))


print(f"\nTotal de columnas sin relación directa: {len(cols_no_coincidentes)}\n")
df_cols_huerfanas = pd.DataFrame(cols_no_coincidentes, columns=["Columna sin relación directa en qname"])
df_cols_huerfanas

Coincidencias de las columnas de 'results' con df_schema['qid']: 0 de 172
Coincidencias de las columnas de 'results' con df_schema['qname']: 126 de 172
Coincidencias de las columnas de 'results' con df_schema['question']: 0 de 172
Coincidencias de las columnas de 'results' con df_schema['type']: 0 de 172
Coincidencias de las columnas de 'results' con df_schema['sub']: 0 de 172
Coincidencias de las columnas de 'results' con df_schema['sq_id']: 0 de 172

Total de columnas sin relación directa: 46



,Columna sin relación directa en qname
0,AIAgentChallengesNeutral
1,AIAgentChallengesSomewhat agree
2,AIAgentChallengesSomewhat disagree
3,AIAgentChallengesStrongly agree
4,AIAgentChallengesStrongly disagree
5,AIAgentImpactNeutral
6,AIAgentImpactSomewhat agree
7,AIAgentImpactSomewhat disagree
8,AIAgentImpactStrongly agree
9,AIAgentImpactStrongly disagree


**Conclusion Del Hallazgo:**

**`qname` es la única clave válida, Las demás columnas (`qid`, `question`, `type`,`sub`,`sq_id`) representan identificadores internos, enunciados largos o metadatos de estructura que no coinciden en formato con los nombres abreviados de las columnas de `results`.**

**Identificación Y Clasificación De Las 46 Columna Que No Tienen Coicidencia:**
 


In [12]:
cols_results = set(df_results.columns)
qnames_schema = set(df_schema["qname"].dropna())
cols_no_coincidentes = sorted(list(cols_results - qnames_schema))

def raiz_mas_larga(col):
    candidatos = [q for q in qnames_schema if col.startswith(q)]  
    return max(candidatos, key=len) if candidatos else None       

def verificar_columna(col):
    if col == "ResponseId":
        return "1. Clave Primaria (PK)", "N/A", "Identificador técnico del encuestado"

    if col == "ConvertedCompYearly":
        return "5. Variable Calculada", "N/A", "Derivada de CompTotal y Currency"

    qname = raiz_mas_larga(col)

    if qname is not None:
        sufijo = col[len(qname):].strip()

        if sufijo in ["HaveWorkedWith", "WantToWorkWith", "Admired"]:
            return "3. Faceta de Tecnología", qname, f'Aspecto: "{sufijo}"'

        elif sufijo in ["Personal use", "Professional use"]:
            return "4. Ámbito de Uso", qname, f'Ámbito: "{sufijo}"'

        else:
            return "2. Opción Likert / Matriz", qname, f'Opción concatenada: "{sufijo}"'

    return "Sin clasificar", "Ninguna", "No se encontró patrón"


resultados = []
for col in cols_no_coincidentes:
    categoria, raiz, detalle = verificar_columna(col)
    resultados.append({
        "Columna en Results": col,
        "Categoría": categoria,
        "Raíz en Schema": raiz,
        "Detalle / Sufijo": detalle
    })

df_verificacion = pd.DataFrame(resultados)

print(f"Total de columnas analizadas: {len(df_verificacion)}")
print("\n--- Distribución por categoría ---")
print(df_verificacion["Categoría"].value_counts())

#display(df_verificacion)

Total de columnas analizadas: 46

--- Distribución por categoría ---
Categoría
3. Faceta de Tecnología      27
2. Opción Likert / Matriz    15
4. Ámbito de Uso              2
5. Variable Calculada         1
1. Clave Primaria (PK)        1
Name: count, dtype: int64


**Al cruzar la base de respuestas con el diccionario de la encuesta, encontramos 46 columnas cuya clave raíz está en el diccionario, pero cuyo nombre exacto en las respuestas sufrió una modificación. Esto ocurre porque representan desgloses o métricas específicas. Las clasificamos funcionalmente en 5 grupos según lo que miden:**

1. **Clave Primaria (1 columna): Es el identificador único e irrepetible asignado a cada desarrollador que completó el formulario (`ResponseId`). Garantiza la trazabilidad del registro de forma anónima dentro del dataset.**

2. **Opciones Likert / Matriz Aplanadas (15 columnas): Corresponden a bloques de preguntas tipo matriz donde se evalúa la adopción, los desafíos o el impacto de la Inteligencia Artificial y herramientas de desarrollo (`AIAgentImpact`, `AIAgentChallenges`,`AITool`). Al exportar la encuesta, cada afirmación y nivel de acuerdo (como `Strongly agree`, `Neutral`, `Currently mostly AI`) se convirtió en una columna individual. Esto permite medir cuantitativamente el grado de aceptación, rechazo o frecuencia con la que los profesionales integran la IA en sus flujos diarios.**

3. **Facetas de Tecnología (27 columnas): En lugar de preguntar por una herramienta en general, la encuesta desglosa cada bloque tecnológico (lenguajes, bases de datos, plataformas, entornos de desarrollo, modelos de IA, etc.) según la relación que el desarrollador tiene con ella. Mide tres aspectos clave del perfil técnico: la experiencia real acumulada durante el último año (`HaveWorkedWith`), el interés de aprendizaje o adopción futura (`WantToWorkWith`) y la percepción de calidad o prestigio que le asigna a esa herramienta (`Admired`).**

4. **Ámbito de Uso (2 columnas): Distingue el entorno de aplicación en el que el desarrollador utiliza sus sistemas operativos (`OpSys`). Separa la elección tecnológica en el ámbito personal (`OpSysPersonal use`) del ámbito laboral (`OpSysProfessional use`). Resulta clave para entender si las tecnologías elegidas obedecen a un gusto personal o a los estándares impuestas por las empresas donde trabajan.**

5. **Variable Calculada de Ingresos (1 columna): Representa una métrica sintética estandarizada por Stack Overflow (`ConvertedCompYearly`). Toma la compensación total declarada por el encuestado (`CompTotal`) y su moneda local (`Currency`) y la convierte a un valor anualizado en dólares estadounidenses (`USD`). Esto habilita la comparación salarial directa entre profesionales de distintos países sin distorsiones por moneda local**

**Vinculacion De `resuls` Con `schema`**

In [ ]:
info = df_schema.drop_duplicates("qname")[["qname", "question", "type"]]

df_atributos = pd.DataFrame({"ColumnaEnResult": df_results.columns})

df_atributos["raiz"] = df_atributos["ColumnaEnResult"].apply(
    lambda c: c if c in qnames_schema else raiz_mas_larga(c)
)

df_atributos = (df_atributos
                .merge(info, left_on="raiz", right_on="qname", how="left")
                .drop(columns="qname"))

df_atributos["dtype"] = df_atributos["ColumnaEnResult"].map(df_results.dtypes.astype(str))

print(df_atributos["type"].value_counts(dropna=False))
print('\nColumnas sin "type":', df_atributos.loc[df_atributos["type"].isna(), "ColumnaEnResult"].tolist())

#df_atributos.head(10)
df_schema.columns

**Se obtuvo una tabla (`df_atributos`) con una fila por cada una de las 172 columnas de `results`, que indica a qué pregunta del `schema` corresponde cada una. Para cada columna se registra:**

- `columna`: nombre en `results`.
- `raiz`: `qname` del `schema` con el que se vincula (exacto o por prefijo).
- `question`: texto completo de la pregunta.
- `type`: tipo de pregunta (`MC = Multiple Choice`, `TE = Texto Libre`, `RO = Orden De Oreferencia` o `Matrix = Tabla De Opciones`). ("El campo type del schema clasifica el formato de la pregunta (MC, TE, RO, Matrix)")
- `dtype`: tipo de dato con el que pandas cargó la columna.

**¿Para qué sirve `df_atributos`?**

1. **Dar contexto inmediato a cada columna:** permite saber en una sola consulta qué se le preguntó exactamente al encuestado (`question`) y con qué formato de pregunta (`type`), sin consultar manualmente `schema.csv`.

2. **Documentar la vinculación completa:** 126 columnas coinciden de forma directa con un `qname` y 44 se vinculan por prefijo (preguntas tipo `Matrix` desglosadas en varias columnas). Solo `ResponseId` y `ConvertedCompYearly` quedan sin pregunta asociada, ya que no forman parte del cuestionario.

3. **Servir de base para la clasificación de atributos:** reúne en un solo lugar el enunciado, el tipo de pregunta y el tipo de dato de pandas (`dtype`). Junto con la inspección de los valores, permite decidir si una variable es cualitativa (nominal u ordinal) o cuantitativa (discreta o continua).

4. **Facilitar el análisis posterior:** al filtrar nulos, agrupar respuestas, graficar o plantear hipótesis, permite consultar rápidamente qué pregunta hay detrás de cada columna y así interpretar mejor sus valores.

Hola

In [13]:
df_results.head(10)

,ResponseId,MainBranch,Age,EdLevel,Employment,EmploymentAddl,WorkExp,LearnCodeChoose,LearnCode,LearnCodeAI,...,AIAgentOrchestration,AIAgentOrchWrite,AIAgentObserveSecure,AIAgentObsWrite,AIAgentExternal,AIAgentExtWrite,AIHuman,AIOpen,ConvertedCompYearly,JobSat
0,1,I am a developer by profession,25-34 years old,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)",Employed,"Caring for dependents (children, elderly, etc.)",8.0,"Yes, I am not new to coding but am learning ne...",Online Courses or Certification (includes all ...,"Yes, I learned how to use AI-enabled tools for...",...,Vertex AI,NaN,NaN,NaN,ChatGPT,NaN,When I don’t trust AI’s answers,"Troubleshooting, profiling, debugging",61256.0,10.0
1,2,I am a developer by profession,25-34 years old,"Associate degree (A.A., A.S., etc.)",Employed,NaN,2.0,"Yes, I am not new to coding but am learning ne...",Online Courses or Certification (includes all ...,"Yes, I learned how to use AI-enabled tools for...",...,NaN,NaN,NaN,NaN,NaN,NaN,When I don’t trust AI’s answers;When I want to...,All skills. AI is a flop.,104413.0,9.0
2,3,I am a developer by profession,35-44 years old,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)","Independent contractor, freelancer, or self-em...",None of the above,10.0,"Yes, I am not new to coding but am learning ne...",Online Courses or Certification (includes all ...,"Yes, I learned how to use AI-enabled tools for...",...,NaN,NaN,NaN,NaN,ChatGPT;Claude Code;GitHub Copilot;Google Gemini,NaN,When I don’t trust AI’s answers;When I want to...,"Understand how things actually work, problem s...",53061.0,8.0
3,4,I am a developer by profession,35-44 years old,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)",Employed,None of the above,4.0,"Yes, I am not new to coding but am learning ne...","Other online resources (e.g. standard search, ...","Yes, I learned how to use AI-enabled tools for...",...,NaN,NaN,NaN,NaN,ChatGPT;Claude Code,NaN,When I don’t trust AI’s answers;When I want to...,NaN,36197.0,6.0
4,5,I am a developer by profession,35-44 years old,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)","Independent contractor, freelancer, or self-em...","Caring for dependents (children, elderly, etc.)",21.0,"No, I am not new to coding and did not learn n...",NaN,"Yes, I learned how to use AI-enabled tools for...",...,NaN,NaN,NaN,NaN,NaN,NaN,When I don’t trust AI’s answers,"critical thinking, the skill to define the tas...",60000.0,7.0
5,6,I am a developer by profession,45-54 years old,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)","Independent contractor, freelancer, or self-em...","Caring for dependents (children, elderly, etc....",15.0,"Yes, I am not new to coding but am learning ne...",Online Courses or Certification (includes all ...,"Yes, I learned how to use AI-enabled tools req...",...,NaN,NaN,NaN,NaN,GitHub Copilot,NaN,When I have ethical or security concerns about...,designing the architecture in the new domain,120000.0,7.0
6,7,I am a developer by profession,25-34 years old,Some college/university study without earning ...,"Independent contractor, freelancer, or self-em...",None of the above,9.0,"Yes, I am not new to coding but am learning ne...",Videos (not associated with specific online co...,"Yes, I learned how to use AI-enabled tools for...",...,NaN,NaN,Sentry,NaN,ChatGPT;Claude Code;Google Gemini,NaN,When I don’t trust AI’s answers;When I need he...,NaN,6240.0,6.0
7,8,I am a developer by profession,35-44 years old,"Professional degree (JD, MD, Ph.D, Ed.D, etc.)",Employed,Engaged in paid work (20-29 hours per week);Tr...,22.0,"Yes, I am not new to coding but am learning ne...",Books / Physical media;Technical documentation...,"Yes, I learned how to use AI-enabled tools for...",...,NaN,NaN,NaN,NaN,NaN,NaN,When I want to learn best practices;When I wan...,"Innovation, system design",72000.0,6.0
8,9,I am a developer by profession,25-34 years old,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)",Employed,"Caring for dependents (children, elderly, etc....",9.0,"Yes, I 